In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import asyncio
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["pandas", "numpy", "matplotlib"]
ASSETS = [{"name": "sensor_data.csv", "url": "/labs/1.1.2/sensor_data.csv", "size": 594794, "sha256": "17497bb76505e4502614230cbba005f589b238eeb031f7559be465ad21df1370"}]

async def _retry(label, operation, attempts=3):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return await operation()
        except Exception as error:
            last_error = error
            if attempt < attempts:
                await asyncio.sleep(0.8 * attempt)
    raise RuntimeError(f"{label}失败（已重试 {attempts} 次）：{last_error}") from last_error

if PACKAGES:
    await _retry("依赖安装", lambda: piplite.install(PACKAGES))

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    async def _download():
        separator = "&" if "?" in asset["url"] else "?"
        response = await pyfetch(f'{asset["url"]}{separator}sha256={asset["sha256"][:16]}')
        if response.status < 200 or response.status >= 300:
            raise RuntimeError(f'HTTP {response.status}：{asset["url"]}')
        return await response.bytes()
    payload = await _retry(f'素材下载 {asset["name"]}', _download)
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 1.1.2 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# 读取数据集 2分
data = _____________

In [ ]:
# 1. 传感器数据统计
# 对传感器类型进行分组，并计算每个组的数据数量和平均值 3分
sensor_stats = _____________(_____________)['Value']._____________
# 输出结果
print("传感器数据数量和平均值:")
print(sensor_stats)

In [ ]:
# 2. 按位置统计温度和湿度数据
# 筛选出温度和湿度数据，然后按位置和传感器类型分组，计算每个组的平均值 2分
location_stats = data[data['SensorType']._____________._____________['Value'].mean().unstack()
# 输出结果
print("每个位置的温度和湿度数据平均值:")
print(location_stats)

In [ ]:
# 3. 数据清洗和异常值处理
# 标记异常值 3分
data['is_abnormal'] = _____________(
    ((_____________) & ((data['Value'] < -10) | (data['Value'] > 50))) |
    ((_____________) & ((data['Value'] < 0) | (data['Value'] > 100))),
    True, False
)
# 输出异常值数量 2分
print("异常值数量:", data['is_abnormal']._____________)
# 填补缺失值
# 使用前向填充和后向填充的方法填补缺失值 4分
data['Value']._____________(_____________, inplace=True)
data['Value']._____________(_____________, inplace=True)
# 保存清洗后的数据
# 删除用于标记异常值的列，并将清洗后的数据保存到新的CSV文件中 4分
cleaned_data = _____________(_____________=['is_abnormal'])
_____________('cleaned_sensor_data.csv', _____________)
print("数据清洗完成，已保存为 'cleaned_sensor_data.csv'")